In [ ]:
import pandas as pd
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from lightgbm import LGBMClassifier


df = pd.read_csv("/Users/aspa/Desktop/abhishek/boi_prototype/datasets/drebin-215-dataset-5560malware-9476-benign.csv")




/var/folders/g6/rc3fpl4n7xj9jnr2zxwjkn900000gn/T/ipykernel_19447/291782140.py:12: DtypeWarning: Columns (0: TelephonyManager.getSimCountryIso) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/aspa/Desktop/abhishek/boi_prototype/datasets/drebin-215-dataset-5560malware-9476-benign.csv")


In [19]:
df["TelephonyManager.getSimCountryIso"]

0        0
1        0
2        0
3        0
4        0
        ..
15031    1
15032    0
15033    0
15034    0
15035    1
Name: TelephonyManager.getSimCountryIso, Length: 15036, dtype: object

In [21]:
# label

y = df["class"]

# features

X = df.drop(
    columns=["class"]
)

# save feature names

feature_names = list(X.columns)

with open(
    "feature_names.json",
    "w"
) as f:

    json.dump(
        feature_names,
        f
    )

# encode labels

encoder = LabelEncoder()

y = encoder.fit_transform(y)

joblib.dump(
    encoder,
    "label_encoder.pkl"
)

# split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert all features to numeric, turning non-numeric strings into NaN if necessary
X_train = X_train.apply(pd.to_numeric, errors='coerce')

# Do the same for your test set
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# model

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

model.fit(
    X_train,
    y_train
)

preds = model.predict(
    X_test
)

print(
    classification_report(
        y_test,
        preds
    )
)

joblib.dump(
    model,
    "malware_model.pkl"
)

[LightGBM] [Info] Number of positive: 4448, number of negative: 7580
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007816 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 425
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 212
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1896
           1       0.98      0.98      0.98      1112

    accuracy                           0.99      3008
   macro avg       0.99      0.99      0.99      3008
weighted avg       0.99      0.99      0.99      3008



['malware_model.pkl']

In [ ]:

MODEL.predict_proba(feature_vector)[0][1]